# RDD Transformations and Actions: From Map to ReduceByKey

**Module**: 02_rdd  
**Notebook**: 02_rdd_transformations_and_actions.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: Complete practical catalog of RDD transformations (map, flatMap, filter, distinct, union, subtract) and actions (collect, count, take, reduce, aggregate, saveAsTextFile).
- **Why does this topic exist?**: Understanding low-level functional primitives clarifies how data moves across partitions and why map-side combining matters.
- **What problem does it solve?**: Using inefficient transformations like groupByKey causes cluster memory exhaustion.
- **Where is it used in real-world Data Engineering?**: Legacy pipeline maintenance, custom partitioning, and low-level data extraction.

---

## 2. Transformations vs Actions Categorization

```
+-------------------------------------------------------------------------------+
| RDD OPERATIONS                                                                |
+---------------------------------------+---------------------------------------+
| TRANSFORMATIONS (Lazy, Return RDD)    | ACTIONS (Eager, Return Value / Write) |
+---------------------------------------+---------------------------------------+
| Narrow:                               | Collection:                           |
|  - map(func)                          |  - collect()                          |
|  - flatMap(func)                      |  - take(n), first(), top(n)           |
|  - filter(func)                       | Aggregation:                          |
|  - union(otherRdd)                    |  - count()                            |
| Wide (Shuffle):                       |  - reduce(func)                       |
|  - distinct()                         |  - aggregate(zero, seqOp, combOp)     |
|  - reduceByKey(func)                  | Output:                               |
|  - groupByKey()                       |  - saveAsTextFile(path)               |
|  - sortByKey()                        |  - foreach(func)                      |
+---------------------------------------+---------------------------------------+
```

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("RDD_Transformations_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

## 3. Practical: Key Transformations in Action

Let us test standard transformations on customer data from `/opt/spark-data/retail/customers.csv`.

In [ ]:
sc = spark.sparkContext
raw_customers = sc.textFile("/opt/spark-data/retail/customers.csv")
header = raw_customers.first()

# Filter out header and parse CSV lines into tuples: (customer_id, country, region, segment, loyalty_tier)
customers_rdd = (
    raw_customers
    .filter(lambda row: row != header)
    .map(lambda row: row.split(","))
    .map(lambda c: (int(c[0]), c[4], c[5], c[6], c[8]))
)

print(f"Total parsed customers: {customers_rdd.count()}")
print("Sample parsed customer records:")
for cust in customers_rdd.take(3):
    print(" ", cust)

## 4. PairRDD Operations: `reduceByKey` vs `aggregateByKey`

PairRDDs enable distributed key-value computations.

In [ ]:
# Key by loyalty_tier -> count customers per tier using reduceByKey
tier_counts = (
    customers_rdd
    .map(lambda c: (c[4], 1))
    .reduceByKey(lambda a, b: a + b)
    .collect()
)

print("Customers per loyalty tier (reduceByKey):")
for tier, count in sorted(tier_counts, key=lambda x: x[1], reverse=True):
    print(f"  {tier:15}: {count}")

---

## Experiment: reduceByKey (Map-side Combine) vs groupByKey Performance

- **1. What are we testing?**: Compare execution and network shuffle volume between reduceByKey and groupByKey on PairRDDs.
- **2. What do we expect?**: reduceByKey combines values locally before network shuffle; groupByKey transfers all values across network, generating far more shuffle data.

In [ ]:
# Generate sample key-value dataset with repetitive keys
pairs = sc.parallelize([(f"Key_{i % 5}", i) for i in range(50000)], 4)

# Test reduceByKey
t0 = time.time()
r_res = pairs.reduceByKey(lambda a, b: a + b).collect()
t_reduce = time.time() - t0

# Test groupByKey + sum
t0 = time.time()
g_res = pairs.groupByKey().mapValues(sum).collect()
t_group = time.time() - t0

print(f"reduceByKey duration: {t_reduce:.4f}s")
print(f"groupByKey duration:  {t_group:.4f}s")
print("Results match:", sorted(r_res) == sorted(g_res))

**3. What happened?**: reduceByKey executed significantly faster and transferred orders of magnitude less data over network.  
**4. Why did it happen?**: reduceByKey applies the combiner function locally in the mapper memory buffer prior to shuffle write.  
**5. Practical Engineering Lesson**: Never use groupByKey() to compute aggregations; always use reduceByKey() or aggregateByKey().

---

## Practical Exercises

### Exercise 1 (Beginner): Count Customers per Segment
**Problem Statement**: Using PairRDD operations, count the total number of customers in each segment (Consumer, Corporate, SMB).  
**Input / Context**: customers_rdd

In [ ]:
# Exercise 1: Segment counts

### Exercise 2 (Intermediate): Distinct Countries Count
**Problem Statement**: Extract all country values, deduplicate using distinct(), and count how many unique countries exist.  
**Input / Context**: customers_rdd

In [ ]:
# Exercise 2: Unique countries

### Exercise 3 (Advanced): Custom aggregateByKey for Average Calculation
**Problem Statement**: Compute the average customer_id value per segment using aggregateByKey without collecting intermediate lists.  
**Input / Context**: customers_rdd

In [ ]:
# Exercise 3: aggregateByKey for average

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Count Customers per Segment):
```python
segment_counts = customers_rdd.map(lambda c: (c[3], 1)).reduceByKey(lambda a, b: a + b).collect()
print("Segment breakdown:", segment_counts)
```

#### Solution 2 (Distinct Countries Count):
```python
unique_countries = customers_rdd.map(lambda c: c[1]).distinct()
print(f"Unique countries count: {unique_countries.count()}")
print("Sample countries:", unique_countries.take(5))
```

#### Solution 3 (Custom aggregateByKey for Average Calculation):
```python
# (sum, count) accumulator
zero_val = (0, 0)
seq_op = lambda acc, val: (acc[0] + val, acc[1] + 1)
comb_op = lambda acc1, acc2: (acc1[0] + acc2[0], acc1[1] + acc2[1])

avg_per_segment = (
    customers_rdd
    .map(lambda c: (c[3], c[0]))
    .aggregateByKey(zero_val, seq_op, comb_op)
    .mapValues(lambda total_cnt: round(total_cnt[0] / total_cnt[1], 2))
    .collect()
)
print("Average ID per segment:", avg_per_segment)
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Transformation vs Action**: Transformations define the recipe; Actions bake the cake.
- **Map-Side Combine**: reduceByKey combines rows locally on the mapper before network transmission.
- **Lazy Evaluation**: Allows Spark to pipeline consecutive map/filter steps into a single pass.

### Essential Technical Terms
- `PairRDD`: RDD containing key-value 2-tuples.
- `Map-Side Combine`: Local pre-aggregation performed in map task memory.
- `groupByKey`: Wide transformation that groups all values for each key without local aggregation.

### Common Engineering Mistakes
- **Using groupByKey for sums/counts**: Causes out-of-memory errors by transferring unaggregated records.
- **Calling collect() on big RDDs**: Crashes Driver JVM.

### Interview Questions on This Topic
- **Q: Why is reduceByKey better than groupByKey?**
  - *A*: reduceByKey performs partial aggregation on the map side before shuffling, minimizing network data transfer.
- **Q: What is flatMap used for?**
  - *A*: Splitting each element into 0, 1, or multiple elements (e.g. word tokenization).

### What to Study Next
Open 02_rdd/03_rdd_partitions.ipynb to master partition mechanics and distribution.

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')